# Energy vs complexity

Part of the code is adapted from https://github.com/ConstanceDws/toolbox_energy

Related paper : https://hal.science/hal-04697137v1

Required install

* pip install codecarbon
* pip install thop
* pip install prettytable


Download the new corpus on arche.

If you have access to a GPU it will be better

In [ ]:
import os
import numpy as np
import random
from torchaudio.transforms import MelSpectrogram
import torchaudio
import librosa
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from tqdm import tqdm
from prettytable import PrettyTable
from codecarbon import EmissionsTracker
from thop import profile, clever_format

np.random.seed(2026)

## Data preprocessing

Use the same data as last week

In [ ]:
data_path = 'data/audio'
     

# Create a dataset with the specified data folder
data_folder = os.path.join(data_path, 'DESED_public_eval_sample')
metadata_file = os.path.join(data_path, 'public_sample.tsv')
# Larger dataset
data_folder = os.path.join(data_path, 'DESED_public_eval/audio/train/')
metadata_file = os.path.join(data_path, 'DESED_public_eval/metadata/train/public_weak.tsv')

print(f"data folder: {data_folder}")
print(f"data folder: {metadata_file}")

In [ ]:
from collections import OrderedDict

# Define the label-to-number mapping dictionary
classes_labels = OrderedDict(
    {
        "Alarm_bell_ringing": 0,
        "Blender": 1,
        "Cat": 2,
        "Dishes": 3,
        "Dog": 4,
        "Electric_shaver_toothbrush": 5,
        "Frying": 6,
        "Running_water": 7,
        "Speech": 8,
        "Vacuum_cleaner": 9,
    }
)

# Create a data loader
batch_size = 8
fs = 44100

target_len = 10 * fs

In [ ]:
import pandas as pd

# Initialize an empty metadata dictionary
metadata = {}

# Read the metadata file (weak.tsv) while skipping the first line (headers)
with open(metadata_file, 'r') as file:
    next(file)  # Skip the first line (headers)
    for line in file:
        parts = line.strip().split()
        filename = parts[0]

        # Extract all event labels from the line
        event_labels = parts[1].split(',')

        # Map each event label to a number and store in a list
        encoded_event_labels = [classes_labels.get(label, -1) for label in event_labels]

        # Add the filename and encoded event labels to the metadata dictionary
        metadata[filename] = encoded_event_labels


In [ ]:
# padding the audio
def pad_audio(waveform):

  if waveform.shape[-1] < target_len:
    waveform = torch.nn.functional.pad(
        waveform, (0, target_len - waveform.shape[-1]), mode="constant")

  elif len(waveform) > target_len:
    rand_onset = random.randint(0, len(waveform) - target_len)
    waveform = waveform[rand_onset:rand_onset + target_len]

  return waveform
     

In [ ]:
def to_mono(waveform):
  if waveform.shape[0] > 1:
    indx = np.random.randint(0, waveform.shape[0] - 1)
    waveform = waveform[indx]
    waveform = waveform.unsqueeze(0)
  return waveform
    

In [ ]:
class WeakDataset(Dataset):
    def __init__(self, data_folder, metadata, target_len, num_frames):
        self.data_folder = data_folder
        self.metadata = metadata
        self.file_list = os.listdir(data_folder)
        self.num_frames = num_frames
        self.transform = MelSpectrogram(
            sample_rate=44100,
            n_fft=2048,
            win_length=2048,
            hop_length=256,
            f_min=0,
            f_max=22050,
            n_mels=128,
            window_fn=torch.hamming_window,
            wkwargs={"periodic": False},
            power=1,
        )
        self.target_len = target_len

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        filename = self.file_list[idx]
        # print(filename)
        file_path = os.path.join(self.data_folder, filename)
        waveform, _ = torchaudio.load(file_path)
        # only one channel
        waveform = to_mono(waveform)

        # pad audio
        waveform = pad_audio(waveform)


        mel_spec = self.transform(waveform)

        event_label = self.metadata[filename]
        #event_label = torch.tensor(event_label, dtype=torch.float32)


        return mel_spec[ :, :, :self.num_frames], event_label[0]
     

## Training methods

In [ ]:
def correct(output, target):
    predicted_digits = output.argmax(1)                            # pick digit with largest network output
    correct_ones = (predicted_digits == target).type(torch.float)  # 1.0 for correct, 0.0 for incorrect
    return correct_ones.sum().item()       

In [ ]:
def train(data_loader, model, criterion, optimizer):
    # Use the GPU (if any)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.train()

    num_batches = len(data_loader)
    num_items = len(data_loader.dataset)

    total_loss = 0
    total_correct = 0
    for data, target in tqdm(data_loader, total=num_batches):
        # Copy data and targets to GPU
        data = data.to(device)
        target = target.to(device)

        # Do a forward pass
        output = model(data)

        # Calculate the loss
        loss = criterion(output, target)
        total_loss += loss

        # Count number of correct digits
        total_correct += correct(output, target)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    train_loss = total_loss/num_batches
    accuracy = total_correct/num_items
    print(f"Average loss: {train_loss:7f}, accuracy: {accuracy:.2%}")
    return train_loss.item(), accuracy

In [ ]:
def test(data_loader, model, criterion, optimizer):
    # Use the GPU (if any)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.train()

    num_batches = len(data_loader)
    num_items = len(data_loader.dataset)

    total_loss = 0
    total_correct = 0
    for data, target in tqdm(data_loader, total=num_batches):
        # Copy data and targets to GPU
        data = data.to(device)
        target = target.to(device)
        with torch.no_grad():
            # Do a forward pass
            output = model(data)

        # Calculate the loss
        loss = criterion(output, target)
        total_loss += loss

        # Count number of correct digits
        total_correct += correct(output, target)


    test_loss = total_loss/num_batches
    accuracy = total_correct/num_items
    print(f"Average loss: {train_loss:7f}, accuracy: {accuracy:.2%}")
    return test_loss.item(), accuracy

## Paramters count

In [ ]:
def count_parameters(model):
    table = PrettyTable(["Modules", "Parameters"])
    total_params = 0
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        params = parameter.numel()
        table.add_row([name, params])
        total_params += params
    print(table)
    print(f"Total Trainable Params: {total_params}")
    return total_params
    

## MLP

In [ ]:
class DynamicMLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(DynamicMLP, self).__init__()

        self.layers = nn.ModuleList([nn.Linear(input_size, hidden_size[0]), nn.ReLU()])

        for i in range(1, num_layers):
            self.layers.extend([nn.Linear(hidden_size[i-1], hidden_size[i]), nn.ReLU()])

        self.layers.extend([nn.Linear(hidden_size[-1], output_size), nn.Softmax(dim=1)])

    def forward(self, x):
        x = x.view(x.size(0), -1)  

        for layer in self.layers:
            x = layer(x)

        return x

In [ ]:
num_frames = 64
hidden_size = [50, 50]
num_layers = 2

MLP = DynamicMLP(input_size= 128*num_frames, hidden_size=hidden_size, output_size=10, num_layers=num_layers)
print(MLP)

In [ ]:
count_parameters(MLP)

In [ ]:
dataset = WeakDataset(data_folder, metadata, target_len, num_frames)
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
examples = enumerate(data_loader)
batch_idx, (example_data, example_targets) = next(examples)

macs, params = profile(MLP, inputs=(example_data,))


macs, params = clever_format([macs, params], "%.3f")
print(f"MACS: {macs} PARAMS:{params}")

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(MLP.parameters(), lr=0.001)

In [ ]:
%%time

# code carbon tracker
tracker_code_carbon = EmissionsTracker(tracking_mode="process", log_level="warning",output_dir=data_path)
tracker_code_carbon.start()

losses = []
accuracies = []
epochs = 4
for epoch in range(epochs):
    print(f"Training epoch: {epoch+1}")
    loss, acc = train(data_loader, MLP, criterion, optimizer)
    losses.append(loss)
    accuracies.append(acc)
    
tracker_code_carbon.stop()

print('Training complete.')

## CNN

In [ ]:
class DynamicCNN(nn.Module):
    def __init__(self, input_channels, output_classes, out_channels, num_layers, num_frame):
        super(DynamicCNN, self).__init__()

        layers = nn.ModuleList([
                nn.Conv2d(input_channels, out_channels[0], kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=(2, 2))
        ])

        for i in range(1, num_layers):
            layers.extend([
                nn.Conv2d(out_channels[i-1], out_channels[i], kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=(2, 2))
            ])

        self.conv_layers = nn.ModuleList(layers)

        fc_input_size = out_channels[-1] * (128 // (2 ** num_layers)) * (num_frame // (2 ** num_layers))

        self.fc = nn.Linear(fc_input_size, output_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        for layer in self.conv_layers:
            x = layer(x)
  
        x = x.view(x.size(0), -1)

        x = self.fc(x)
        x = self.softmax(x)
        return x


In [ ]:
num_frames = 64
num_layers = 2
out_channels = [32, 16]

CNN = DynamicCNN(input_channels = 1, output_classes= 10, out_channels=out_channels, num_layers=num_layers, num_frame=num_frames)
print(CNN)

In [ ]:
count_parameters(CNN)

In [ ]:
examples = enumerate(data_loader)
batch_idx, (example_data, example_targets) = next(examples)

macs, params = profile(CNN, inputs=(example_data,))

macs, params = clever_format([macs, params], "%.3f")
print(f"MACS: {macs} PARAMS:{params}")

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(CNN.parameters(), lr=0.001)

In [ ]:
%%time

# code carbon tracker
tracker_code_carbon = EmissionsTracker(tracking_mode="process", log_level="warning",output_dir=data_path)
tracker_code_carbon.start()

losses = []
accuracies = []
epochs = 4
for epoch in range(epochs):
    print(f"Training epoch: {epoch+1}")
    loss, acc = train(data_loader, CNN, criterion, optimizer)
    losses.append(loss)
    accuracies.append(acc)
    
tracker_code_carbon.stop()

print('Training complete.')

## RNN

In [ ]:
class DynamicRNN(nn.Module):
    def __init__(self, input_size, output_size, hidden_size, num_layers, num_directions=1):
        super(DynamicRNN, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.num_directions = num_directions
        

        self.rnn = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, bidirectional=(num_directions == 2))

        fc_input_size = hidden_size * num_directions * 64
        self.fc = nn.Linear(fc_input_size, output_size)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        batch_size = x.size(0)
        sequence_length = x.size(3) 

        x = x.view(batch_size, sequence_length, -1)
        h0 = torch.zeros(self.num_layers * self.num_directions, batch_size, self.hidden_size).to(x.device)

        out, _ = self.rnn(x, h0)
        out = out.contiguous().view(batch_size, -1)

        out = self.softmax(self.fc(out))

        return out


In [ ]:
hidden_size = 50
num_layers = 2

RNN = DynamicRNN(input_size = 128, output_size=10, hidden_size=hidden_size, num_layers=num_layers)
print(RNN)

In [ ]:
count_parameters(RNN)

In [ ]:
examples = enumerate(data_loader)
batch_idx, (example_data, example_targets) = next(examples)

macs, params = profile(RNN, inputs=(example_data,))

macs, params = clever_format([macs, params], "%.3f")
print(f"MACS: {macs} PARAMS:{params}")

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(RNN.parameters(), lr=0.001)

In [ ]:
%%time

# code carbon tracker
tracker_code_carbon = EmissionsTracker(tracking_mode="process", log_level="warning",output_dir=data_path)
tracker_code_carbon.start()

losses = []
accuracies = []
epochs = 4
for epoch in range(epochs):
    print(f"Training epoch: {epoch+1}")
    loss, acc = train(data_loader, RNN, criterion, optimizer)
    losses.append(loss)
    accuracies.append(acc)
    
tracker_code_carbon.stop()

print('Training complete.')

## CRNN

In [ ]:
class DynamicCRNN(nn.Module):
    def __init__(self, input_channels, output_classes, out_channels, hidden_size_rnn, num_layers, num_directions=1):
        super(DynamicCRNN, self).__init__()

        self.num_layers_cnn = num_layers[0]
        self.num_layers_rnn = num_layers[1]

        layers = nn.ModuleList([
                nn.Conv2d(input_channels, out_channels[0], kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=(2, 2))
        ])

        for i in range(1, self.num_layers_cnn):
            layers.extend([
                nn.Conv2d(out_channels[i-1], out_channels[i], kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=(2, 2))
            ])
            
        self.hidden_size_rnn = hidden_size_rnn
        self.num_directions = num_directions
        self.conv_layers = nn.ModuleList(layers)
        input_size = out_channels[-1] * (128 // (2 ** num_layers[0]))
        self.rnn = nn.GRU(input_size=input_size, hidden_size=self.hidden_size_rnn, num_layers=self.num_layers_rnn, batch_first=True, bidirectional=(num_directions == 2))
        self.fc = nn.Linear(self.hidden_size_rnn*(num_frames // (2 ** self.num_layers_cnn)), output_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        for layer in self.conv_layers:
            x = layer(x)

        x = x.view(x.size(0), x.size(3), -1)      
        x, _ = self.rnn(x)
        x = x.contiguous().view(-1, self.hidden_size_rnn*(num_frames // (2 ** self.num_layers_cnn)))
        x = self.fc(x)
        x = self.softmax(x)
        return x

In [ ]:
num_frames = 64
hidden_size = 50
out_channels = [32, 16]
num_layers = [2, 1]

CRNN = DynamicCRNN(input_channels = 1, output_classes=10, out_channels=out_channels, hidden_size_rnn=hidden_size, num_layers=num_layers)
print(CRNN)

In [ ]:
count_parameters(CRNN)

In [ ]:
examples = enumerate(data_loader)
batch_idx, (example_data, example_targets) = next(examples)

macs, params = profile(CRNN, inputs=(example_data,))

macs, params = clever_format([macs, params], "%.3f")
print(f"MACS: {macs} PARAMS:{params}")

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(CRNN.parameters(), lr=0.001)

In [ ]:
%%time



# code carbon tracker
tracker_code_carbon = EmissionsTracker(tracking_mode="process", log_level="warning",output_dir=data_path)
tracker_code_carbon.start()

losses = []
accuracies = []
epochs = 4
for epoch in range(epochs):
    print(f"Training epoch: {epoch+1}")
    loss, acc = train(data_loader, CRNN, criterion, optimizer)
    losses.append(loss)
    accuracies.append(acc)
    
tracker_code_carbon.stop()

print('Training complete.')

## Todo
(note that you might need to increase models size to see interesting things)
* Is there any simple relation between parameters/MACs/consumption
    * Compare the values you obtain with other people in the room
* What does it says about measuring the energy consumption?
* Compute the carbon intensity factor
    * Compare it with other people (also local vs. cloud)
* Compare the energy consumption with "process" and with "machine" as `tracking mode`
    * Explain the differences
    * We'll stick to "process" from now on
* compare train and test energy co
* Change the batch size
    * What is the impact?
* Change the network hyperparameters such that they all have the same number of parameter
    * Compute the MACs and energy comsumption
    * Are they the same for all the networks?
* For each model
    * Evaluate the parameters/MACS/energy consumption for at least 5 different model versions (change the number of parameters)
    * Plot energy vs paramter, energy vs MACS, MACS vs parameter
* Compare the versions obtained for each model type
    * Is there is simple relation between these metrics, regardless of the method architecture?
        * You can try to plot all the models on the same graph
* In depth analysis for the energy consumption:
    * split the energy between componenets (CPU, RAM, GPU) and plot